# LAB 05 - Bài 21: Long-range dependency

Huấn luyện Simple RNN trên ACL IMDb. Sau đó giữ nguyên ngữ cảnh và độ dài câu, chỉ chuyển cụm sentiment quan trọng (`excellent` / `terrible`) đến đầu, giữa hoặc cuối chuỗi.


In [1]:
# ============================================================
# BÀI 21 - EXPERIMENT 6: LONG-RANGE DEPENDENCY
# Tải ACL IMDb, tạo train/validation/test cố định và tokenize.
# ============================================================

from pathlib import Path
import tarfile
import urllib.request
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

DATA_URL = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
DATA_DIR = Path("/content/aclImdb")
ARCHIVE_PATH = Path("/content/aclImdb_v1.tar.gz")

if not (DATA_DIR / "train" / "pos").exists():
    if not ARCHIVE_PATH.exists():
        print("Đang tải ACL IMDb từ Stanford (~84 MB)...")
        urllib.request.urlretrieve(DATA_URL, ARCHIVE_PATH)
    print("Đang giải nén dataset...")
    with tarfile.open(ARCHIVE_PATH, "r:gz") as archive:
        archive.extractall(path="/content")

def read_split(split):
    texts, labels = [], []
    for label_name, label_id in (("neg", 0), ("pos", 1)):
        folder = DATA_DIR / split / label_name
        for file_path in sorted(folder.glob("*.txt")):
            texts.append(file_path.read_text(encoding="utf-8", errors="replace").strip())
            labels.append(label_id)
    return texts, np.asarray(labels, dtype=np.int32)

all_train_texts, all_train_labels = read_split("train")
test_texts, y_test = read_split("test")
train_texts, val_texts, y_train, y_val = train_test_split(
    all_train_texts,
    all_train_labels,
    test_size=0.1,
    random_state=42,
    stratify=all_train_labels,
)

MAX_VOCAB_SIZE = 20_000
MAX_LEN = 200
EMBEDDING_DIM = 64
HIDDEN_SIZE = 64
BATCH_SIZE = 64
EPOCHS = 3
SEED = 42

tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(train_texts)
vocab_size = min(MAX_VOCAB_SIZE, len(tokenizer.word_index) + 1)

X_train = pad_sequences(tokenizer.texts_to_sequences(train_texts), maxlen=MAX_LEN,
                        padding="post", truncating="post")
X_val = pad_sequences(tokenizer.texts_to_sequences(val_texts), maxlen=MAX_LEN,
                      padding="post", truncating="post")
X_test = pad_sequences(tokenizer.texts_to_sequences(test_texts), maxlen=MAX_LEN,
                       padding="post", truncating="post")

print("Vocabulary size:", vocab_size)
print("Train/validation/test:", X_train.shape, X_val.shape, X_test.shape)


Đang tải ACL IMDb từ Stanford (~84 MB)...
Đang giải nén dataset...


/tmp/ipykernel_975/2416857981.py:26: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extractall(path="/content")


Vocabulary size: 20000
Train/validation/test: (22500, 200) (2500, 200) (25000, 200)


In [2]:
# ============================================================
# BÀI 21 - HUẤN LUYỆN SIMPLE RNN
# Giữ một kiến trúc cố định để quan sát dự đoán theo vị trí cue.
# ============================================================

import time
import tensorflow as tf
from tensorflow.keras import layers, models

tf.keras.utils.set_random_seed(SEED)
rnn_model = models.Sequential([
    layers.Input(shape=(MAX_LEN,)),
    layers.Embedding(vocab_size, EMBEDDING_DIM, mask_zero=True),
    layers.SimpleRNN(HIDDEN_SIZE),
    layers.Dense(1, activation="sigmoid"),
])
rnn_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

start_time = time.perf_counter()
rnn_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=2,
)
training_time = time.perf_counter() - start_time

test_probabilities = rnn_model.predict(X_test, batch_size=BATCH_SIZE, verbose=0).ravel()
test_predictions = (test_probabilities >= 0.5).astype("int32")
print("\nTest accuracy:", round(accuracy_score(y_test, test_predictions), 4))
print("Test F1:", round(f1_score(y_test, test_predictions, zero_division=0), 4))
print("Training time (seconds):", round(training_time, 2))


Epoch 1/3
352/352 - 38s - 108ms/step - accuracy: 0.5916 - loss: 0.6646 - val_accuracy: 0.5936 - val_loss: 0.6629
Epoch 2/3
352/352 - 33s - 93ms/step - accuracy: 0.7584 - loss: 0.5125 - val_accuracy: 0.7856 - val_loss: 0.4868
Epoch 3/3
352/352 - 34s - 98ms/step - accuracy: 0.8198 - loss: 0.4110 - val_accuracy: 0.8000 - val_loss: 0.4699

Test accuracy: 0.7946
Test F1: 0.7901
Training time (seconds): 111.81


In [3]:
# ============================================================
# BÀI 21 - SO SÁNH TỪ KHÓA Ở ĐẦU / GIỮA / CUỐI CHUỖI
# Các câu có cùng ngữ cảnh trung tính và cùng độ dài; chỉ đổi
# vị trí cụm sentiment quan trọng.
# ============================================================

import pandas as pd

neutral_phrase = (
    "the scene shows actors and camera while the story moves across the room "
    "and music plays in the background dialogue continues as the setting "
    "changes with more scenes and people on screen"
)
neutral_tokens = neutral_phrase.split()
BASE_LENGTH = 190
base_tokens = (neutral_tokens * ((BASE_LENGTH // len(neutral_tokens)) + 1))[:BASE_LENGTH]

# Chèn cùng cụm từ ở ba vị trí. Tổng số token của mọi câu bằng nhau.
insert_positions = {
    "beginning": 3,
    "middle": len(base_tokens) // 2,
    "end": len(base_tokens) - 2,
}

def make_controlled_review(sentiment_word, position_name):
    tokens = base_tokens.copy()
    cue = ["the", "movie", "was", sentiment_word]
    insert_at = insert_positions[position_name]
    tokens[insert_at:insert_at] = cue
    return " ".join(tokens)

examples = []
for sentiment_word, true_label in (("excellent", 1), ("terrible", 0)):
    for position_name in ("beginning", "middle", "end"):
        examples.append({
            "true_sentiment": "positive" if true_label == 1 else "negative",
            "position": position_name,
            "text": make_controlled_review(sentiment_word, position_name),
        })

example_texts = [row["text"] for row in examples]
example_sequences = tokenizer.texts_to_sequences(example_texts)
X_examples = pad_sequences(
    example_sequences, maxlen=MAX_LEN, padding="post", truncating="post"
)

prob_positive = rnn_model.predict(X_examples, verbose=0).ravel()
for row, probability in zip(examples, prob_positive):
    row["p_positive"] = float(probability)
    row["predicted_sentiment"] = "positive" if probability >= 0.5 else "negative"
    row["correct"] = row["predicted_sentiment"] == row["true_sentiment"]
    row["sequence_tokens"] = len(tokenizer.texts_to_sequences([row["text"]])[0])

results_df = pd.DataFrame(examples)[[
    "true_sentiment", "position", "sequence_tokens",
    "p_positive", "predicted_sentiment", "correct"
]]
print("Các câu kiểm soát có cùng độ dài:")
print(results_df.to_string(index=False, formatters={"p_positive": "{:.4f}".format}))

# Trung bình xác suất positive theo vị trí và sentiment thật.
summary_df = results_df.pivot_table(
    index="position", columns="true_sentiment", values="p_positive", aggfunc="mean"
).reindex(["beginning", "middle", "end"])
print("\nTrung bình xác suất positive theo vị trí:")
print(summary_df.round(4))

OUTPUT_CSV = "/content/lab05_bai21_position_results.csv"
results_df.to_csv(OUTPUT_CSV, index=False)
print("\nĐã lưu kết quả:", OUTPUT_CSV)

# Diễn giải: đây là quan sát trên các câu kiểm soát và một model.
# Nó KHÔNG chứng minh vanishing gradient. Kết quả còn chịu ảnh hưởng
# bởi dữ liệu huấn luyện, từ vựng, kiến trúc, seed và cách tạo câu.


Các câu kiểm soát có cùng độ dài:
true_sentiment  position  sequence_tokens p_positive predicted_sentiment  correct
      positive beginning              194     0.9392            positive     True
      positive    middle              194     0.9458            positive     True
      positive       end              194     0.9465            positive     True
      negative beginning              194     0.9383            positive    False
      negative    middle              194     0.9460            positive    False
      negative       end              194     0.7350            positive    False

Trung bình xác suất positive theo vị trí:
true_sentiment  negative  positive
position                          
beginning         0.9383    0.9392
middle            0.9460    0.9458
end               0.7350    0.9465

Đã lưu kết quả: /content/lab05_bai21_position_results.csv
